# Importing Libraries

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import missingno as msno
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import plotly.io as pio
pio.renderers.default = 'notebook'

import warnings
warnings.filterwarnings('ignore')

# Loading Data

In [ ]:
df = pd.read_csv('..\data\churn.csv')

# Understanding Data

In [ ]:
df.head(5)

In [ ]:
df.shape

In [ ]:
df.info

In [ ]:
df.columns.values

In [ ]:
df.dtypes

# Cleaning Data

In [ ]:
# Visualise missing values as matrix
msno.matrix(df);
print("No peculiar pattern. No missing data.")

In [ ]:
# Remove CustomerID
df = df.drop(['customerID'], axis = 1)
df.head(5)

In [ ]:
# Checking for Null Values in data
df['TotalCharges'] = pd.to_numeric(df.TotalCharges, errors = 'coerce')
print("Some indirect missingness in data.")
df.isnull().sum()

In [ ]:
print(" TotalCharges has 11 missing values")
df[np.isnan(df['TotalCharges'])]

In [ ]:
# Fill missing values in TotalCharges with mean of TotalCharges column
df.fillna(df['TotalCharges'].mean())

In [ ]:
df.isnull().sum()

In [ ]:
# Missing Values in tenure column
print("Tenure column is 0 for these entries even though the MonthlyCharges column is not empty.")
df[df['tenure'] == 0].index

In [ ]:
# Delete rows with missing values in tenure column
df.drop(labels = df[df['tenure'] == 0].index, axis = 0, inplace = True)
df[df['tenure'] == 0].index

In [ ]:
# Viewing internet service column
df["InternetService"].describe(include = ['object', 'bool'])

In [ ]:
# Viewing numerical columns
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[numerical_cols].describe()

In [ ]:
df = df.drop(columns=['tenure group'])

In [ ]:
# Run this cell after Visualising Data
df['Churn'] = df['Churn'].map({'No':0,'Yes':1})

# Visualising Data

## Churn Distribution

In [ ]:
fig = px.histogram(df, x='Churn',
                   title='Churn Distribution',
                   color='Churn',
                   color_discrete_map={'No':'#2ecc71', 'Yes':'#e74c3c'}) 
fig.update_layout(xaxis_title='Churn', yaxis_title='Number of Customers')
fig.show()

In [ ]:
fig = px.pie(df, names='Churn', 
             title='Churn Distribution',
             color='Churn',
             color_discrete_map={'No': '#2ecc71', 'Yes': '#e74c3c'})
fig.show()

## Churn Rate by Contract Type

In [ ]:
churn_contract = df.groupby(['Contract', 'Churn']).size().reset_index(name='Count')

In [ ]:
fig = px.bar(churn_contract,
             x='Contract',
             y='Count',
             color='Churn',
             title='Churn Rate by Contract Type',
             color_discrete_map={'No': '#2ecc71', 'Yes': '#e74c3c'},
             barmode='group')
fig.update_layout(xaxis_title='Contract Type',
                  yaxis_title='Number of Customers',
                  legend_title='Churn',
                  template='plotly_white')
fig.show()

- Customers on month-to-month contracts churn the most, with ~1,650 churned customers, nearly as many as those who stayed (~2,200).
- In contrast, customers on one-year and two-year contracts are far more loyal.
- Churn drops dramatically, with two-year contract customers showing almost zero churn (~50 churned vs ~1,630 retained)
-  Month-to-month customers are significantly more likely to leave, while long-term contract customers show very high retention, suggesting that locking customers into longer contracts is an effective retention strategy.

## Churn By Monthly Charges

In [ ]:
fig = px.box(df, x='Churn',
             y='MonthlyCharges',
             color='Churn',
             title='Monthly Charges by Churn',
             color_discrete_map={'No': '#2ecc71', 'Yes': '#e74c3c'})
fig.update_layout(xaxis_title='Churn',
                  yaxis_title='Monthly Charges',
                  legend_title='Churn',
                  template='plotly_white')
fig.show()

- Customers who churned tend to have a higher median monthly charge (80 dollars) compared to customers who stayed (median-65 dollars).
- The churned group also has a tighter spread. Most churned customers are paying between (57–94 dollars), while retained customers have a much wider range (25–89 dollars), suggesting the company retains customers across all price points but loses more of the higher-paying ones.
- Higher monthly charges are associated with increased churn risk
- Customers paying above $65/month are more likely to leave, suggesting pricing may be a key driver of customer attrition.

## Tenure by Churn

In [ ]:
fig = px.histogram(df, x='tenure',
                   color='Churn',
                   barmode='group')
fig.update_layout(xaxis_title='Tenure',
                  yaxis_title='Number of Customers',
                  legend_title='Churn',
                  template='plotly_white')
fig.show()

- Churn is heavily concentrated among new customers (0–10 months tenure)
- Newer customers are far more likely to leave.
- As tenure increases, the churn rate shrink significantly and almost disappear by 50+ months.
- Meanwhile the customer who stayed grow and dominate the right side, with a big spike at 70+ months showing long-term customers are extremely loyal.
- Customers in their first 10 months are at the highest risk of leaving.
- If the business can retain customers past the 1-year mark, they are significantly more likely to become long-term loyal customers.
- This suggests early engagement and onboarding strategies are critical.

## Correlation Heatmap

In [ ]:
corr_matrix = df[['tenure', 'MonthlyCharges', 'TotalCharges']].corr()
corr_matrix

In [ ]:
fig = go.Figure(go.Heatmap(z=corr_matrix.values, 
                 x=corr_matrix.columns.tolist(),
                 y=corr_matrix.columns.tolist(),
                 colorscale='RdBu',
                 zmin=0,zmax=1))
fig.update_layout(title='Correlation Heatmap',
                  template='plotly_white')
fig.show()

- Tenure vs TotalCharges (0.83) — very strong correlation. The longer a customer stays, the more total they've paid over time.
- MonthlyCharges vs TotalCharges (0.65) — moderate-strong correlation. Higher monthly bills naturally lead to higher total charges.
- tenure vs MonthlyCharges (0.25) — weak correlation. How long someone has been a customer has little relationship with how much they pay monthly.
- TotalCharges is heavily influenced by tenure, meaning it may not add much independent predictive power to the model beyond what tenure already captures.

## Encoding Categorical Columns

In [ ]:
df = pd.get_dummies(df, drop_first=True)
print(df.shape)
print(df.dtypes)

In [ ]:
df = df.astype({col: int for col in df.select_dtypes(bool).columns})
print(df.dtypes)

## Scale Numerical Features

In [ ]:
scaler = StandardScaler()
df[['tenure', 'MonthlyCharges', 'TotalCharges']] = scaler.fit_transform(df[['tenure', 'MonthlyCharges', 'TotalCharges']])
data_path = os.path.join(os.getcwd(), '..', 'data', 'scaler.pkl')
joblib.dump(scaler, data_path)
print(f"Scalar saved to: {data_path}")

In [ ]:
print(df[['tenure','MonthlyCharges','TotalCharges']].describe())

## Split the Data

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, Y_train, Y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape, Y_train.shape, X_test.shape, Y_test.shape)

In [ ]:
X_train.to_csv(os.path.join(os.getcwd(), '..', 'data', 'X_train.csv'), index=False)
X_test.to_csv(os.path.join(os.getcwd(), '..', 'data', 'X_test.csv'), index=False)
Y_train.to_csv(os.path.join(os.getcwd(), '..', 'data', 'Y_train.csv'), index=False)
Y_test.to_csv(os.path.join(os.getcwd(), '..', 'data', 'Y_test.csv'), index=False)